# AI image and video enlarger

Post-process existing **FastH3, original H3, Wan, Qwen or uploaded media** using
Real-ESRGAN. The default is native **2×** (`RealESRGAN_x2plus`); the optional
4× anime model can produce either 4× or 2× with a final Lanczos downsample.

Select a T4 GPU for the notebook. Setup preserves the installed PyTorch stack,
loads only the small upscaler, and downloads only the selected weight with its
recorded SHA-256. The video path streams one frame at a time with tiled inference;
it keeps the source constant frame rate and audio. This is frame-wise super-resolution;
inspect the result in motion for flicker. See [the guide](../doc/AI_UPSCALER.md)
for validation scope and limitations.

Run **A once**, then edit and rerun **B** for each file. Upload through Colab's Files
panel or use an existing `/content/...` output. With B copied into an existing
generation notebook after A, an empty `INPUT_PATH` uses that notebook's `OUTPUT_PATH`.
The original file stays intact; each result receives a unique name and a JSON receipt.
Copy/download results before disconnecting the runtime. This notebook does not allocate
or release Colab sessions on your behalf.


In [ ]:
AI_UPSCALER_SOURCES = {'media_upscale.py': '#!/usr/bin/env python3\n"""Standalone tiled AI enlargement. Heavy dependencies are imported only for inference."""\nfrom __future__ import annotations\n\nimport argparse\nfrom fractions import Fraction\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport urllib.request\n\nROOT = Path(__file__).resolve().parents[1]\nCATALOG = Path(__file__).with_name("media_upscale_models.json")\nDEFAULT_MODEL_DIR = ROOT / "artifacts/media-upscale/models"\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef load_catalog(path=CATALOG):\n    data = json.loads(Path(path).read_text())\n    if data.get("schema_version") != 1 or not isinstance(data.get("models"), dict):\n        raise ValueError("Unsupported model catalog schema")\n    for name, entry in data["models"].items():\n        required = {"file", "url", "sha256", "size", "scale", "license", "source"}\n        if not required.issubset(entry):\n            raise ValueError(f"Incomplete catalog entry: {name}")\n        if Path(entry["file"]).name != entry["file"] or entry["file"] in ("", ".", ".."):\n            raise ValueError(f"Unsafe model filename: {name}")\n        if entry["scale"] not in (2, 4) or not isinstance(entry["size"], int) or entry["size"] <= 0:\n            raise ValueError(f"Invalid model size/scale: {name}")\n        if len(entry["sha256"]) != 64 or any(c not in "0123456789abcdef" for c in entry["sha256"]):\n            raise ValueError(f"Invalid model digest: {name}")\n        if not entry["url"].startswith("https://"):\n            raise ValueError(f"Model download requires HTTPS: {name}")\n    return data["models"]\n\n\ndef verify_model(path, entry):\n    if Path(path).stat().st_size != entry["size"] or sha256(path) != entry["sha256"]:\n        raise ValueError(f"Model size/hash mismatch: {path}; remove the invalid cache and download again")\n\n\ndef download_model(entry, model_dir):\n    directory = Path(model_dir)\n    directory.mkdir(parents=True, exist_ok=True)\n    destination = directory / entry["file"]\n    if destination.exists():\n        verify_model(destination, entry)\n        return destination\n    fd, temporary = tempfile.mkstemp(prefix=".model-", suffix=".part", dir=directory)\n    try:\n        with os.fdopen(fd, "wb") as output, urllib.request.urlopen(entry["url"], timeout=60) as response:\n            total = 0\n            while True:\n                chunk = response.read(1024 * 1024)\n                if not chunk:\n                    break\n                total += len(chunk)\n                if total > entry["size"]:\n                    raise ValueError("Downloaded model exceeds catalog size")\n                output.write(chunk)\n        verify_model(temporary, entry)\n        # Publish without overwriting a cache another process may have created.\n        try:\n            os.link(temporary, destination)\n        except FileExistsError:\n            verify_model(destination, entry)\n        return destination\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n\n\ndef load_descriptor(path):\n    import torch\n    from spandrel import ImageModelDescriptor, ModelLoader\n    state = torch.load(path, map_location="cpu", weights_only=True)\n    for key in ("params_ema", "params", "state_dict"):\n        if isinstance(state, dict) and isinstance(state.get(key), dict):\n            state = state[key]\n            break\n    descriptor = ModelLoader(device="cpu").load_from_state_dict(state)\n    if not isinstance(descriptor, ImageModelDescriptor):\n        raise ValueError("Weights are not a spandrel image model")\n    return descriptor\n\n\nclass TileEngine:\n    """Keep only one padded tile on the inference device; output stays in CPU RAM."""\n    def __init__(self, descriptor, native_scale, scale=2, tile=192, tile_pad=16,\n                 device="auto", precision="auto"):\n        import torch\n        if scale not in (2, 4) or scale > native_scale:\n            raise ValueError("Requested scale must be 2/4 and cannot exceed the model\'s native AI scale")\n        if tile <= 0 or tile_pad < 0:\n            raise ValueError("tile must be positive and tile-pad nonnegative")\n        if descriptor.scale != native_scale or descriptor.input_channels != 3 or descriptor.output_channels != 3:\n            raise ValueError("Model descriptor scale/channels do not match catalog RGB contract")\n        if device == "auto":\n            device = "cuda" if torch.cuda.is_available() else "cpu"\n        if device not in ("cpu", "cuda") or device == "cuda" and not torch.cuda.is_available():\n            raise ValueError("Requested CUDA device is unavailable")\n        if precision == "auto":\n            precision = "fp16" if device == "cuda" and descriptor.supports_half else "fp32"\n        if precision not in ("fp32", "fp16"):\n            raise ValueError("Unsupported precision")\n        if precision == "fp16" and (device != "cuda" or not descriptor.supports_half):\n            raise ValueError("FP16 requires CUDA and a model supporting half precision; use fp32")\n        self.descriptor = descriptor.to(device).eval()\n        if precision == "fp16":\n            self.descriptor.half()\n        else:\n            self.descriptor.float()\n        self.device, self.precision = device, precision\n        self.native_scale, self.scale = native_scale, scale\n        self.tile, self.tile_pad = tile, tile_pad\n        self.frames = self.tiles = 0\n        self.inference_seconds = 0.0\n\n    def upscale(self, rgb):\n        import numpy as np\n        import torch\n        from PIL import Image\n        if rgb.dtype != np.uint8 or rgb.ndim != 3 or rgb.shape[2] != 3:\n            raise ValueError("Tile engine requires RGB uint8 pixels")\n        height, width, _ = rgb.shape\n        native = self.native_scale\n        output = np.empty((height * native, width * native, 3), dtype=np.uint8)\n        started = time.perf_counter()\n        with torch.inference_mode():\n            for top in range(0, height, self.tile):\n                bottom = min(top + self.tile, height)\n                for left in range(0, width, self.tile):\n                    right = min(left + self.tile, width)\n                    x0, x1 = max(0, left - self.tile_pad), min(width, right + self.tile_pad)\n                    y0, y1 = max(0, top - self.tile_pad), min(height, bottom + self.tile_pad)\n                    pixels = np.ascontiguousarray(rgb[y0:y1, x0:x1].transpose(2, 0, 1))\n                    tensor = torch.from_numpy(pixels).unsqueeze(0).to(\n                        device=self.device, dtype=torch.float16 if self.precision == "fp16" else torch.float32)\n                    tensor.div_(255)\n                    result = self.descriptor(tensor)\n                    expected = (1, 3, (y1 - y0) * native, (x1 - x0) * native)\n                    if tuple(result.shape) != expected or not torch.isfinite(result).all().item():\n                        raise ValueError("Model returned invalid tile dimensions or nonfinite pixels")\n                    crop = result[0, :, (top-y0)*native:(bottom-y0)*native,\n                                  (left-x0)*native:(right-x0)*native]\n                    crop = crop.clamp(0, 1).mul(255).round().to(torch.uint8).cpu().numpy().transpose(1, 2, 0)\n                    output[top*native:bottom*native, left*native:right*native] = crop\n                    self.tiles += 1\n                    del tensor, result, crop\n        self.inference_seconds += time.perf_counter() - started\n        self.frames += 1\n        if self.scale != native:\n            output = np.asarray(Image.fromarray(output).resize(\n                (width*self.scale, height*self.scale), Image.Resampling.LANCZOS))\n        return output\n\n\ndef check_paths(source, output):\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if not source.is_file():\n        raise FileNotFoundError(source)\n    if source == output:\n        raise ValueError("Input and output must be different paths")\n    receipt = Path(str(output) + ".upscale.json")\n    if output.exists() or receipt.exists():\n        raise FileExistsError(f"Refusing to overwrite output or receipt: {output}")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    return source, output\n\n\ndef temporary_output(output):\n    fd, name = tempfile.mkstemp(prefix=".upscale-", suffix=output.suffix, dir=output.parent)\n    os.close(fd)\n    return Path(name)\n\n\ndef publish(temporary, output, receipt):\n    sidecar = Path(str(output) + ".upscale.json")\n    fd, name = tempfile.mkstemp(prefix=".receipt-", suffix=".json", dir=output.parent)\n    try:\n        with os.fdopen(fd, "w") as handle:\n            json.dump(receipt, handle, indent=2)\n            handle.write("\\n")\n        os.link(name, sidecar)\n        try:\n            os.link(temporary, output)\n        except BaseException:\n            sidecar.unlink()\n            raise\n    finally:\n        Path(name).unlink(missing_ok=True)\n\n\ndef base_receipt(source, temporary, output, engine, model_id, entry, input_sha, started):\n    if sha256(source) != input_sha:\n        raise ValueError("Source changed while processing; result was not published")\n    return dict(schema_version=1, status="complete", input=dict(path=str(source), sha256=input_sha),\n                output=dict(path=str(output), sha256=sha256(temporary)),\n                model=dict(id=model_id, sha256=entry["sha256"], native_scale=engine.native_scale,\n                           requested_scale=engine.scale, license=entry["license"], source=entry["source"]),\n                device=engine.device, precision=engine.precision, tile=engine.tile, tile_pad=engine.tile_pad,\n                inference=dict(frames=engine.frames, tiles=engine.tiles, seconds=engine.inference_seconds),\n                elapsed_seconds=time.perf_counter()-started)\n\n\ndef upscale_image(source, output, engine, model_id, entry):\n    import numpy as np\n    from PIL import Image, ImageOps\n    source, output = check_paths(source, output)\n    if output.suffix.lower() != ".png":\n        raise ValueError("Image output must be PNG")\n    started, input_sha = time.perf_counter(), sha256(source)\n    temporary = temporary_output(output)\n    try:\n        with Image.open(source) as encoded_image:\n            if getattr(encoded_image, "n_frames", 1) != 1:\n                raise ValueError("Animated images are unsupported; choose a single frame explicitly")\n            encoded_size = encoded_image.size\n            exif_orientation = encoded_image.getexif().get(274, 1)\n            image = ImageOps.exif_transpose(encoded_image)\n            # Pillow may expose 16-bit RGB PNG as RGB, so inspect its IHDR bit depth too.\n            with source.open("rb") as handle:\n                header = handle.read(26)\n            if (header[:8] == b"\\x89PNG\\r\\n\\x1a\\n" and header[24] == 16) or image.mode not in ("RGB", "RGBA", "L", "LA", "P"):\n                raise ValueError("16-bit/high-depth or unsupported image mode; convert explicitly to 8-bit RGB/RGBA")\n            image.load()\n            alpha = image.convert("RGBA").getchannel("A") if image.mode in ("RGBA", "LA") or "transparency" in image.info else None\n            result = Image.fromarray(engine.upscale(np.asarray(image.convert("RGB"))))\n            if alpha is not None:\n                result.putalpha(alpha.resize(result.size, Image.Resampling.LANCZOS))\n            result.save(temporary, format="PNG")\n            expected_size, expected_mode = result.size, result.mode\n        with Image.open(temporary) as verified:\n            verified.load()\n            if verified.size != expected_size or verified.mode != expected_mode:\n                raise ValueError("PNG validation failed")\n        receipt = base_receipt(source, temporary, output, engine, model_id, entry, input_sha, started)\n        receipt.update(media=dict(kind="image", input_width=image.width, input_height=image.height,\n                                 encoded_input_width=encoded_size[0], encoded_input_height=encoded_size[1],\n                                 input_exif_orientation=exif_orientation,\n                                 orientation_normalized=exif_orientation in range(2, 9),\n                                 output_width=expected_size[0], output_height=expected_size[1], mode=expected_mode),\n                       validation=dict(complete_decode=True, dimensions=True, source_unchanged=True,\n                                       scope="Technical checks only; image quality has not been reviewed"))\n        publish(temporary, output, receipt)\n        return receipt\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef require_ffmpeg():\n    for program in ("ffmpeg", "ffprobe"):\n        if not shutil.which(program):\n            raise RuntimeError(f"{program} is required for video")\n\n\ndef probe_video(path):\n    require_ffmpeg()\n    command = ["ffprobe", "-v", "error", "-count_frames", "-show_streams", "-show_format", "-of", "json", str(path)]\n    result = subprocess.run(command, capture_output=True, text=True)\n    if result.returncode or result.stderr.strip():\n        raise ValueError(f"ffprobe failed: {result.stderr.strip()}")\n    return json.loads(result.stdout)\n\n\ndef video_stream(info):\n    streams = [s for s in info["streams"] if s["codec_type"] == "video"]\n    if len(streams) != 1:\n        raise ValueError("Exactly one video stream is required")\n    return streams[0]\n\n\ndef scan_cfr(path, fps, expected_count, time_base=Fraction(1, 1000000)):\n    """Check every timestamp with bounded memory; avg_frame_rate alone cannot prove CFR."""\n    with tempfile.TemporaryFile() as errors:\n        process = subprocess.Popen(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_frames",\n                                    "-show_entries", "frame=best_effort_timestamp_time", "-of", "csv=p=0", str(path)],\n                                   stdout=subprocess.PIPE, stderr=errors, text=True)\n        count, first = 0, None\n        try:\n            for line in process.stdout:\n                value = line.strip().split(",")[0]\n                if not value:\n                    continue\n                try:\n                    stamp = float(value)\n                except ValueError as error:\n                    raise ValueError("Video is missing frame timestamps") from error\n                if not math.isfinite(stamp):\n                    raise ValueError("Video has invalid frame timestamps")\n                if first is None:\n                    first = stamp\n                if abs(stamp - (first + count / float(fps))) > max(0.000003, float(time_base) * 1.01):\n                    raise ValueError("VFR input is unsupported; convert explicitly to CFR before enlargement")\n                count += 1\n            if process.wait() != 0:\n                errors.seek(0)\n                raise ValueError(f"Frame timestamp probe failed: {errors.read().decode(errors=\'replace\')}")\n            if count != expected_count or count == 0:\n                raise ValueError("Frame timestamp count does not match complete probe decode")\n            return first\n        finally:\n            process.stdout.close()\n            stop_process(process)\n\n\ndef validate_input_video(path, info):\n    video = video_stream(info)\n    if video.get("color_transfer") in ("smpte2084", "arib-std-b67") or video.get("color_primaries") == "bt2020" or int(video.get("bits_per_raw_sample") or 0) > 8 or any(\n            marker in video.get("pix_fmt", "") for marker in ("9", "10", "12", "14", "16", "48", "64", "f32")):\n        raise ValueError("HDR/high-depth video is unsupported; tone-map explicitly to 8-bit SDR first")\n    rotation = float(video.get("tags", {}).get("rotate", 0))\n    for side in video.get("side_data_list", []):\n        rotation = float(side.get("rotation", rotation))\n    if rotation != 0:\n        raise ValueError("Nonzero video rotation is unsupported; bake rotation into pixels first")\n    if Fraction(video.get("sample_aspect_ratio", "1:1").replace(":", "/")) != 1:\n        raise ValueError("Non-square video pixels are unsupported; convert explicitly to square pixels first")\n    try:\n        fps = Fraction(video["avg_frame_rate"])\n        count = int(video["nb_read_frames"])\n    except (KeyError, ValueError, ZeroDivisionError) as error:\n        raise ValueError("Video requires a valid rational FPS and complete decoded frame count") from error\n    if fps <= 0 or count <= 0:\n        raise ValueError("Video has no decodable frames or valid FPS")\n    time_base = Fraction(video["time_base"])\n    first = scan_cfr(path, fps, count, time_base)\n    if abs(stream_duration(video) - count / float(fps)) > max(0.000003, float(time_base) * 2):\n        raise ValueError("Input video duration is not an exact CFR frame interval; conform explicitly before enlargement")\n    return video, fps, count, first\n\n\ndef read_frame(pipe, size):\n    """A pipe read may be short; EOF midway through a frame is an error."""\n    data = bytearray(size)\n    offset = 0\n    while offset < size:\n        chunk = pipe.read(size-offset)\n        if not chunk:\n            if offset:\n                raise ValueError("Decoder returned a partial RGB frame")\n            return None\n        data[offset:offset+len(chunk)] = chunk\n        offset += len(chunk)\n    return data\n\n\ndef stop_process(process):\n    if process is not None and process.poll() is None:\n        process.terminate()\n        try:\n            process.wait(timeout=5)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait()\n\n\ndef stream_frames(source, temporary, engine, video, fps, expected_count, first, source_info):\n    import numpy as np\n    width, height = int(video["width"]), int(video["height"])\n    out_width, out_height = width*engine.scale, height*engine.scale\n    decoder = encoder = None\n    with tempfile.TemporaryFile() as decode_errors, tempfile.TemporaryFile() as encode_errors:\n        try:\n            decoder = subprocess.Popen(["ffmpeg", "-v", "error", "-xerror", "-noautorotate", "-i", str(source),\n                                        "-map", "0:v:0", "-an", "-sn", "-dn", "-fps_mode", "passthrough",\n                                        "-f", "rawvideo", "-pix_fmt", "rgb24", "pipe:1"],\n                                       stdout=subprocess.PIPE, stderr=decode_errors)\n            command = ["ffmpeg", "-v", "error", "-xerror", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",\n                       "-video_size", f"{out_width}x{out_height}", "-framerate", str(fps), "-i", "pipe:0",\n                       "-copyts", "-itsoffset", str(-first), "-i", str(source), "-map", "0:v:0", "-map", "1:a?",\n                       "-map_metadata", "-1", "-c:v", "libx264", "-crf", "18", "-preset", "medium", "-pix_fmt", "yuv420p",\n                       "-fps_mode", "passthrough", "-video_track_timescale", str(fps.numerator)]\n            # AAC is MP4-compatible; transcode other source audio explicitly.\n            audio = [s for s in source_info["streams"] if s["codec_type"] == "audio"]\n            for index, stream in enumerate(audio):\n                command += [f"-c:a:{index}", "copy" if stream["codec_name"] == "aac" else "aac"]\n            command += ["-movflags", "+faststart", str(temporary)]\n            encoder = subprocess.Popen(command, stdin=subprocess.PIPE, stderr=encode_errors)\n            count = 0\n            last_progress = time.perf_counter()\n            while True:\n                frame = read_frame(decoder.stdout, width*height*3)\n                if frame is None:\n                    break\n                pixels = np.frombuffer(frame, dtype=np.uint8).reshape(height, width, 3)\n                enlarged = engine.upscale(pixels)\n                encoder.stdin.write(enlarged.tobytes())\n                count += 1\n                now = time.perf_counter()\n                if count == 1 or count == expected_count or now - last_progress >= 5:\n                    print(f"Upscaled frames: {count}/{expected_count}", file=sys.stderr, flush=True)\n                    last_progress = now\n                del frame, pixels, enlarged\n            decoder.stdout.close()\n            if decoder.wait() != 0:\n                decode_errors.seek(0)\n                raise ValueError(f"Complete input decode failed: {decode_errors.read().decode(errors=\'replace\')}")\n            if count != expected_count:\n                raise ValueError(f"Input decoded frame count mismatch: expected {expected_count}, got {count}")\n            encoder.stdin.close()\n            if encoder.wait() != 0:\n                encode_errors.seek(0)\n                raise ValueError(f"Video encoder failed: {encode_errors.read().decode(errors=\'replace\')}")\n            return count\n        except BrokenPipeError as error:\n            encode_errors.seek(0)\n            raise ValueError(f"Video encoder pipe failed: {encode_errors.read().decode(errors=\'replace\')}") from error\n        finally:\n            # Stop both children before closing pipes (which might otherwise block on flush).\n            stop_process(decoder)\n            stop_process(encoder)\n            if decoder and decoder.stdout and not decoder.stdout.closed:\n                decoder.stdout.close()\n            if encoder and encoder.stdin and not encoder.stdin.closed:\n                try:\n                    encoder.stdin.close()\n                except BrokenPipeError:\n                    pass\n\n\ndef stream_duration(stream):\n    if "duration" in stream:\n        return float(stream["duration"])\n    # Matroska stores duration in stream tags rather than the MP4 duration field.\n    tagged = stream.get("tags", {}).get("DURATION")\n    if tagged:\n        hours, minutes, seconds = tagged.split(":")\n        return int(hours)*3600 + int(minutes)*60 + float(seconds)\n    raise ValueError("Cannot verify source/output stream duration")\n\n\ndef validate_output_video(source_info, output_info, scale, fps, count, first=0):\n    source, output = video_stream(source_info), video_stream(output_info)\n    if (output["width"], output["height"]) != (source["width"]*scale, source["height"]*scale):\n        raise ValueError("Output dimensions do not match requested scale")\n    if Fraction(output["avg_frame_rate"]) != fps or int(output["nb_read_frames"]) != count:\n        raise ValueError("Output FPS/frame count mismatch")\n    expected_duration = count / float(fps)\n    if abs(float(output["duration"]) - expected_duration) > 0.0001:\n        raise ValueError("Output video duration mismatch")\n    source_audio = [s for s in source_info["streams"] if s["codec_type"] == "audio"]\n    output_audio = [s for s in output_info["streams"] if s["codec_type"] == "audio"]\n    if len(source_audio) != len(output_audio):\n        raise ValueError("Output audio stream count mismatch")\n    for before, after in zip(source_audio, output_audio):\n        if after.get("codec_name") != "aac" or int(after["channels"]) != int(before["channels"]):\n            raise ValueError("Output audio codec/channel mismatch")\n        if abs(stream_duration(before)-stream_duration(after)) > 0.075:\n            raise ValueError("Output audio duration mismatch")\n        expected_start = float(before.get("start_time", 0)) - first\n        if abs(float(after.get("start_time", 0)) - expected_start) > 0.075:\n            raise ValueError("Output audio/video synchronization offset mismatch")\n        if before["codec_name"] == "aac" and int(before["nb_read_frames"]) != int(after["nb_read_frames"]):\n            raise ValueError("Copied AAC audio decoded frame count mismatch")\n    if abs(float(source_info["format"]["duration"])-float(output_info["format"]["duration"])) > max(0.075, 1/float(fps)):\n        raise ValueError("Output container duration mismatch")\n\n\ndef upscale_video(source, output, engine, model_id, entry):\n    source, output = check_paths(source, output)\n    if output.suffix.lower() != ".mp4":\n        raise ValueError("Video output must be MP4")\n    started, input_sha = time.perf_counter(), sha256(source)\n    source_info = probe_video(source)\n    video, fps, count, first = validate_input_video(source, source_info)\n    temporary = temporary_output(output)\n    try:\n        stream_frames(source, temporary, engine, video, fps, count, first, source_info)\n        output_info = probe_video(temporary)\n        validate_output_video(source_info, output_info, engine.scale, fps, count, first)\n        scan_cfr(temporary, fps, count, Fraction(video_stream(output_info)["time_base"]))\n        # Decode every output video/audio packet locally; metadata checks alone are insufficient.\n        complete = subprocess.run(["ffmpeg", "-v", "error", "-xerror", "-i", str(temporary),\n                                   "-map", "0:v", "-map", "0:a?", "-f", "null", "-"], capture_output=True)\n        if complete.returncode:\n            raise ValueError(f"Complete output decode failed: {complete.stderr.decode(errors=\'replace\')}")\n        output_info["format"]["filename"] = str(output)\n        receipt = base_receipt(source, temporary, output, engine, model_id, entry, input_sha, started)\n        receipt.update(media=dict(kind="video", fps=str(fps), frames=count, input=source_info, output=output_info),\n                       validation=dict(complete_input_decode=True, complete_output_decode=True, dimensions=True,\n                                       fps=True, frame_count=True, frame_order="sequential decode/inference/encode; no interpolation",\n                                       audio=True, durations=True, source_unchanged=True,\n                                       scope="Technical checks only; temporal consistency and perceptual quality are not proven"))\n        publish(temporary, output, receipt)\n        return receipt\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef parser():\n    shared = argparse.ArgumentParser(add_help=False)\n    shared.add_argument("--model", default="realesrgan-x2plus")\n    shared.add_argument("--model-dir", type=Path, default=DEFAULT_MODEL_DIR)\n    shared.add_argument("--scale", type=int, choices=(2, 4), default=2)\n    shared.add_argument("--tile", type=int, default=192)\n    shared.add_argument("--tile-pad", type=int, default=16)\n    shared.add_argument("--device", choices=("auto", "cuda", "cpu"), default="auto")\n    shared.add_argument("--precision", choices=("auto", "fp32", "fp16"), default="auto")\n    result = argparse.ArgumentParser(description=__doc__)\n    commands = result.add_subparsers(dest="command", required=True)\n    commands.add_parser("download", parents=[shared], help="Download and verify only the selected model")\n    for command in ("image", "video"):\n        child = commands.add_parser(command, parents=[shared])\n        child.add_argument("input", type=Path)\n        child.add_argument("--output", required=True, type=Path)\n    return result\n\n\ndef main(argv=None):\n    args = parser().parse_args(argv)\n    try:\n        models = load_catalog()\n        if args.model not in models:\n            raise ValueError(f"Unknown model {args.model!r}; choose from {\', \'.join(models)}")\n        entry = models[args.model]\n        if args.command == "download":\n            print(download_model(entry, args.model_dir))\n            return 0\n        check_paths(args.input, args.output)\n        if args.scale > entry["scale"]:\n            raise ValueError("Requested factor exceeds model\'s native AI scale; select a native x4 model")\n        model_path = args.model_dir / entry["file"]\n        if not model_path.is_file():\n            raise FileNotFoundError(f"Model is missing: {model_path}; run the download subcommand first")\n        verify_model(model_path, entry)\n        engine = TileEngine(load_descriptor(model_path), entry["scale"], args.scale,\n                            args.tile, args.tile_pad, args.device, args.precision)\n        function = upscale_image if args.command == "image" else upscale_video\n        receipt = function(args.input, args.output, engine, args.model, entry)\n        print(json.dumps(receipt, indent=2))\n        return 0\n    except (ValueError, RuntimeError, OSError, ImportError) as error:\n        print(f"media-upscale: {error}", file=sys.stderr)\n        return 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'media_upscale_models.json': '{\n  "schema_version": 1,\n  "checked_date": "2026-10-11",\n  "loader": "spandrel==0.4.2",\n  "hash_provenance": "SHA-256 calculated from the official GitHub release bytes; byte counts cross-checked against the GitHub releases API.",\n  "models": {\n    "realesrgan-x2plus": {\n      "file": "RealESRGAN_x2plus.pth",\n      "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",\n      "sha256": "49fafd45f8fd7aa8d31ab2a22d14d91b536c34494a5cfe31eb5d89c2fa266abb",\n      "size": 67061725,\n      "scale": 2,\n      "architecture": "RRDBNet, 23 blocks, 64 features, growth 32",\n      "license": "weights-not-separately-specified",\n      "source_license": "BSD-3-Clause",\n      "source": "https://github.com/xinntao/Real-ESRGAN/blob/master/docs/model_zoo.md"\n    },\n    "realesrgan-x4plus": {\n      "file": "RealESRGAN_x4plus.pth",\n      "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",\n      "sha256": "4fa0d38905f75ac06eb49a7951b426670021be3018265fd191d2125df9d682f1",\n      "size": 67040989,\n      "scale": 4,\n      "architecture": "RRDBNet, 23 blocks, 64 features, growth 32",\n      "license": "weights-not-separately-specified",\n      "source_license": "BSD-3-Clause",\n      "source": "https://github.com/xinntao/Real-ESRGAN/blob/master/docs/model_zoo.md"\n    },\n    "realesrgan-anime-x4": {\n      "file": "RealESRGAN_x4plus_anime_6B.pth",\n      "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",\n      "sha256": "f872d837d3c90ed2e05227bed711af5671a6fd1c9f7d7e91c911a61f155e99da",\n      "size": 17938799,\n      "scale": 4,\n      "architecture": "RRDBNet, 6 blocks, 64 features, growth 32",\n      "license": "weights-not-separately-specified",\n      "source_license": "BSD-3-Clause",\n      "source": "https://github.com/xinntao/Real-ESRGAN/blob/master/docs/model_zoo.md"\n    }\n  }\n}\n'}
# A — install the small inference dependency and restore the embedded toolkit
import importlib.util, json, shutil, subprocess, sys
from pathlib import Path

TOOLKIT = Path("/content/media-upscale-toolkit")
TOOLKIT.mkdir(parents=True, exist_ok=True)
for filename, source in AI_UPSCALER_SOURCES.items():
    path = TOOLKIT / filename
    if not path.is_file() or path.read_text(encoding="utf-8") != source:
        path.write_text(source, encoding="utf-8")

import torch
from importlib.metadata import version
torch_before = torch.__version__
torch_distribution_before = version("torch")
subprocess.run([sys.executable, "-m", "pip", "install", "spandrel==0.4.2", "Pillow>=10.0"], check=True)
if version("torch") != torch_distribution_before:
    raise RuntimeError("The PyTorch installation changed. Restart the runtime before continuing.")
import spandrel
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("ffmpeg and ffprobe are required; use a standard Colab runtime.")
print(json.dumps({"torch": torch_before, "spandrel": version("spandrel"),
                  "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                  "toolkit": str(TOOLKIT)}, indent=2))


In [ ]:
# B — choose a generated file or an uploaded image/video
MEDIA_TYPE = "video" # @param ["image", "video"]
INPUT_PATH = "" # @param {type:"string"}
MODEL = "realesrgan-x2plus" # @param ["realesrgan-x2plus", "realesrgan-x4plus", "realesrgan-anime-x4"]
SCALE = 2 # @param [2, 4]
TILE = 192 # @param {type:"integer"}
TILE_PAD = 16 # @param {type:"integer"}
PRECISION = "auto" # @param ["auto", "fp32", "fp16"]
OUTPUT_DIR = "/content/ai-upscale-output" # @param {type:"string"}

from datetime import datetime, timezone
import uuid

source_value = INPUT_PATH.strip() or globals().get("OUTPUT_PATH")
if not source_value:
    raise ValueError("Set INPUT_PATH to an uploaded/generated file, or run generation to set OUTPUT_PATH.")
source = Path(source_value).resolve()
if not source.is_file():
    raise FileNotFoundError(source)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
suffix = ".mp4" if MEDIA_TYPE == "video" else ".png"
destination = Path(OUTPUT_DIR) / f"{source.stem}_{SCALE}x_{stamp}{suffix}"
subprocess.run([sys.executable, str(TOOLKIT / "media_upscale.py"), "download",
                "--model", MODEL, "--model-dir", "/content/ai-upscale-models"], check=True)
command = [sys.executable, str(TOOLKIT / "media_upscale.py"), MEDIA_TYPE,
           str(source), "--output", str(destination), "--model", MODEL,
           "--model-dir", "/content/ai-upscale-models", "--scale", str(SCALE),
           "--tile", str(TILE), "--tile-pad", str(TILE_PAD),
           "--device", "auto", "--precision", PRECISION]
subprocess.run(command, check=True)
receipt_path = Path(str(destination) + ".upscale.json")
receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
if receipt.get("status") != "complete":
    raise RuntimeError("Upscaling did not produce a complete receipt")
UPSCALED_OUTPUT_PATH = destination
print("UPSCALED_OUTPUT_PATH=", destination)
print("RECEIPT_PATH=", receipt_path)
from IPython.display import Image as DisplayImage, Video, display
display(Video(str(destination), embed=True, width=720) if MEDIA_TYPE == "video"
        else DisplayImage(filename=str(destination), width=720))
